In [ ]:
"""
Kaggle setup.

This notebook is self-contained: every class/function it needs (DMADataset,
the fused-cost BCE objective, DynamicWeightXGB, DynamicWeightSklearn, ...) is
defined in the cells below in the SAME kernel, so there is no dependency on
importing the bytetrack repo / yolox package. All that's needed as input is
the pre-generated DMA .npz feature files (produced locally by
`python -m yolox.DMA.generate_data`, see yolox/DMA/generate_data.py).

Before running:
  1. Zip and upload the relevant datasets/<name>_dma_train and
     datasets/<name>_dma_val folders (or datasets/*-dma*.zip, already
     prepared) as a private Kaggle Dataset, e.g. "dma-mot17" / "dma-sportmot".
  2. Add Data -> attach that dataset to this notebook.
  3. Update TRAIN_DIR / VAL_DIR below to match the attached path under
     /kaggle/input/.
"""

!pip install -q optuna

import shlex
from pathlib import Path

# --- Point these at wherever you attached the Kaggle Dataset(s) ---
DATASET = "mot17"  # "mot17" or "sportmot"

_INPUT_DIRS = {
    "mot17":    ("/kaggle/input/dma-mot17/mot17_dma_train", "/kaggle/input/dma-mot17/mot17_dma_val"),
    "sportmot": ("/kaggle/input/dma-sportmot/sportmot_train", "/kaggle/input/dma-sportmot/sportmot_val"),
}
TRAIN_DIR, VAL_DIR = _INPUT_DIRS[DATASET]

# Matches the feature subset used by tools/run_dma_ml_ablation.sh (see
# yolox/DMA/features.py for index -> name layout); set to None/"" for all 15.
FEATURE_INDICES = "1,2,3,7,10,13"

OUT_ROOT = Path(f"/kaggle/working/dma_ml_ablation/{DATASET}")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {DATASET}")
print(f"Train dir: {TRAIN_DIR}  ({len(list(Path(TRAIN_DIR).glob('*.npz')))} files)")
print(f"Val dir:   {VAL_DIR}  ({len(list(Path(VAL_DIR).glob('*.npz')))} files)")
print(f"Output:    {OUT_ROOT}")


# XGB

In [ ]:
"""
PyTorch Dataset for training DynamicWeightNet.

Each sample is a (feature_vector, label) pair where:
  feature_vector: np.ndarray (FEAT_DIM,)
  label:          float  1.0 = correct match (same GT identity)
                         0.0 = wrong match   (different GT identity)

Training .npz files are generated by generate_data.py and contain:
  features:         (N, FEAT_DIM)  float32
  labels:           (N,)           float32  {0, 1}
  motion_costs:     (N,)           float32  IoU-based motion cost per pair
  appearance_costs: (N,)           float32  cosine distance per pair
"""

from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset


class DMADataset(Dataset):
    def __init__(
        self,
        data_paths: List[str],
        normalize: bool = True,
        pos_neg_ratio: Optional[float] = None,
        feature_indices: Optional[List[int]] = None,
    ):
        """
        Args:
            data_paths:      list of .npz file paths produced by generate_data.py
            normalize:       z-score normalise features (stats computed from data)
            pos_neg_ratio:   if set, downsample negatives so pos:neg = 1:ratio
            feature_indices: if set, keep only these columns of the stored
                              (FEAT_DIM,) feature vector, in this order - lets
                              you ablate features.py's feature set without
                              re-running generate_data.py (see
                              analyze_features.py for which indices to drop).
        """
        feats, labels, m_costs, a_costs = [], [], [], []
        for p in data_paths:
            d = np.load(p)
            feats.append(d["features"].astype(np.float32))
            labels.append(d["labels"].astype(np.float32))
            if "motion_costs" in d:
                m_costs.append(d["motion_costs"].astype(np.float32))
            if "appearance_costs" in d:
                a_costs.append(d["appearance_costs"].astype(np.float32))

        self.features = np.concatenate(feats, axis=0)
        if feature_indices is not None:
            self.features = self.features[:, feature_indices]
        self.labels = np.concatenate(labels, axis=0)
        self.motion_costs = (
            np.concatenate(m_costs, axis=0) if m_costs else None
        )
        self.appearance_costs = (
            np.concatenate(a_costs, axis=0) if a_costs else None
        )

        if pos_neg_ratio is not None:
            self._balance(pos_neg_ratio)

        if normalize:
            self.mean = self.features.mean(axis=0)
            self.std = self.features.std(axis=0) + 1e-6
            self.features = (self.features - self.mean) / self.std
        else:
            self.mean = np.zeros(self.features.shape[1], dtype=np.float32)
            self.std = np.ones(self.features.shape[1], dtype=np.float32)

    def _balance(self, ratio: float):
        pos_idx = np.where(self.labels == 1)[0]
        neg_idx = np.where(self.labels == 0)[0]
        n_neg_keep = int(len(pos_idx) * ratio)
        if n_neg_keep < len(neg_idx):
            neg_idx = neg_idx[
                np.random.choice(len(neg_idx), n_neg_keep, replace=False)
            ]
        keep = np.sort(np.concatenate([pos_idx, neg_idx]))
        self.features = self.features[keep]
        self.labels = self.labels[keep]
        if self.motion_costs is not None:
            self.motion_costs = self.motion_costs[keep]
        if self.appearance_costs is not None:
            self.appearance_costs = self.appearance_costs[keep]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        feat = torch.from_numpy(self.features[idx])
        label = torch.tensor(self.labels[idx])
        # Return raw (unnormalized) costs for use in the loss function
        mc = torch.tensor(
            self.motion_costs[idx] if self.motion_costs is not None
            else self.features[idx][0]  # fallback (less accurate); idx 0 = motion_cost
        )
        ac = torch.tensor(
            self.appearance_costs[idx] if self.appearance_costs is not None
            else self.features[idx][5]  # idx 5 = cosine_dist
        )
        return feat, label, mc, ac

    def get_normalization_stats(self) -> dict:
        return {
            "mean": self.mean.tolist(),
            "std": self.std.tolist(),
        }

    def class_balance(self) -> Tuple[int, int]:
        pos = int(self.labels.sum())
        neg = len(self.labels) - pos
        return pos, neg

    @staticmethod
    def split(
        data_paths: List[str],
        val_ratio: float = 0.1,
        normalize: bool = True,
        pos_neg_ratio: Optional[float] = None,
        seed: int = 42,
        feature_indices: Optional[List[int]] = None,
    ) -> Tuple["DMADataset", "DMADataset"]:
        """Split paths into train/val datasets."""
        rng = np.random.default_rng(seed)
        paths = list(data_paths)
        rng.shuffle(paths)
        n_val = max(1, int(len(paths) * val_ratio))
        val_paths = paths[:n_val]
        train_paths = paths[n_val:]
        train_ds = DMADataset(
            train_paths, normalize=normalize, pos_neg_ratio=pos_neg_ratio,
            feature_indices=feature_indices,
        )
        # Re-use train stats for val normalisation
        val_ds = DMADataset(val_paths, normalize=False, feature_indices=feature_indices)
        val_ds.features = (val_ds.features - train_ds.mean) / train_ds.std
        val_ds.mean = train_ds.mean
        val_ds.std = train_ds.std
        return train_ds, val_ds


In [ ]:
"""
Custom LightGBM objective/eval mirroring train.py's BCEWeightedLoss.

The booster predicts a single raw score z (margin). w_motion = sigmoid(z),
w_reid = 1 - w_motion (equivalent to the 2-logit softmax in DynamicWeightNet,
since only the logit difference matters for a 2-class softmax).

fused = w_motion * motion_cost + w_reid * appearance_cost
      = appearance_cost + sigmoid(z) * (motion_cost - appearance_cost)

target = 1 - label   (label=1 correct match -> want fused low -> target=0)
Loss   = BCE(fused, target)

Gradient/hessian are derived by the chain rule through sigmoid(z) so that
LightGBM's Newton boosting optimizes the same fused-cost BCE objective the
MLP is trained on, rather than a generic classification loss.
"""

import numpy as np


def _sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30.0, 30.0)))


def make_bce_fused_objective(motion_cost: np.ndarray, appearance_cost: np.ndarray, labels: np.ndarray):
    """
    Returns a LightGBM-compatible fobj(preds, dataset) -> (grad, hess).

    motion_cost / appearance_cost / labels must be aligned row-for-row with
    the lgb.Dataset this objective will be used to train (same order, same
    length) — LightGBM does not reorder rows across boosting rounds.
    """
    target = (1.0 - labels).astype(np.float64)
    mc = motion_cost.astype(np.float64)
    ac = appearance_cost.astype(np.float64)
    d = mc - ac

    def objective(preds: np.ndarray, train_data) -> tuple:
        z = preds.astype(np.float64)
        s = _sigmoid(z)
        f = np.clip(ac + s * d, 1e-6, 1.0 - 1e-6)
        t = target

        D = f * (1.0 - f)
        N = f - t
        dLdf = N / D
        d2Ldf2 = (D - N * (1.0 - 2.0 * f)) / (D ** 2)

        sp = s * (1.0 - s)            # ds/dz
        spp = sp * (1.0 - 2.0 * s)    # d2s/dz2

        dfdz = d * sp
        d2fdz2 = d * spp

        grad = dLdf * dfdz
        hess = d2Ldf2 * (dfdz ** 2) + dLdf * d2fdz2
        hess = np.clip(hess, 1e-3, None)  # keep Newton step well-defined
        return grad, hess

    return objective


def make_fused_eval(motion_cost: np.ndarray, appearance_cost: np.ndarray, labels: np.ndarray, name: str = "val_f1"):
    """
    Returns a LightGBM-compatible feval(preds, dataset) -> (name, value, is_higher_better)
    reporting F1 on the fused-cost < 0.5 decision, matching evaluate() in train.py.
    """
    mc = motion_cost.astype(np.float64)
    ac = appearance_cost.astype(np.float64)
    lbl = labels.astype(np.float64)

    def feval(preds: np.ndarray, train_data) -> tuple:
        z = preds.astype(np.float64)
        s = _sigmoid(z)
        fused = np.clip(ac + s * (mc - ac), 1e-6, 1.0 - 1e-6)
        pred_match = (fused < 0.5).astype(np.float64)

        tp = np.sum((pred_match == 1) & (lbl == 1))
        fp = np.sum((pred_match == 1) & (lbl == 0))
        fn = np.sum((pred_match == 0) & (lbl == 1))
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        return name, float(f1), True

    return feval


In [ ]:
"""
XGBoost alternative DynamicWeightNet.

Same external interface as DynamicWeightNet / DynamicWeightGBM so drop-in
replacement inside DMAFusion:
  - predict_numpy(x_np) -> (N, 2) [w_motion, w_reid], rows sum 1
  - save(path, stats) / load(path)

Booster trained on single raw margin z via the exact same fused-cost BCE
custom objective as DynamicWeightGBM (see gbm_objective.py); [w_motion,
w_reid] = [sigmoid(z), 1 - sigmoid(z)].
"""

import pickle

import numpy as np
import xgboost as xgb


def _sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30.0, 30.0)))


class DynamicWeightXGB:
    """Wraps an xgboost.Booster predicting a single raw margin score."""

    def __init__(self, booster: xgb.Booster):
        self.booster = booster

    # torch-style no-ops so DMAFusion can treat this like DynamicWeightNet
    def to(self, device):
        return self

    def eval(self):
        return self

    def predict_numpy(self, x_np: np.ndarray) -> np.ndarray:
        dmat = xgb.DMatrix(x_np)
        z = self.booster.predict(dmat, output_margin=True)
        w_motion = _sigmoid(z)
        w_reid = 1.0 - w_motion
        return np.stack([w_motion, w_reid], axis=1).astype(np.float32)

    def save(self, path: str, stats: dict = None):
        payload = {"model_str": self.booster.save_raw(raw_format="json"), "stats": stats}
        with open(path, "wb") as f:
            pickle.dump(payload, f)

    @classmethod
    def load(cls, path: str):
        with open(path, "rb") as f:
            payload = pickle.load(f)
        booster = xgb.Booster()
        booster.load_model(bytearray(payload["model_str"]))
        return cls(booster), payload.get("stats")


In [ ]:
XGB_TUNE_ARGS = (
    f"--data-dir {TRAIN_DIR} --val-data-dir {VAL_DIR} "
    f"--out-dir {OUT_ROOT}/xgb "
    f"--n-trials 50 --num-boost-round 2000 --early-stopping 50 "
    f"--feature-indices {FEATURE_INDICES}"
)


In [ ]:
"""
Bayesian hyperparameter search (Optuna, TPE) for DynamicWeightXGB.

Uses the exact same data pipeline and fused-cost BCE objective as
train_xgb.py -- only the XGBoost hyperparameters are searched, the
loss/target formulation stays identical so results stay comparable with
tune_gbm.py.

CLI usage (outside Kaggle):
  python -m yolox.DMA.tune_xgb \\
    --data-dir  datasets/mot17_dma \\
    --out-dir   weights/dma_xgb_mot17 \\
    --n-trials  50

In this notebook, main() takes an explicit argv list instead of reading
sys.argv (Kaggle's kernel launcher args would otherwise get parsed as
positional/unknown arguments) -- see XGB_TUNE_ARGS in the cell above.
"""

import argparse
import json

import optuna
import xgboost as xgb


def _as_xgb_feval(lgb_style_feval):
    """Adapt a (preds, data) -> (name, value, is_higher_better) feval (LightGBM
    convention, used by gbm_objective.make_fused_eval) to xgboost's
    custom_metric convention: (preds, dmatrix) -> (name, value)."""

    def feval(preds, dmatrix):
        name, value, _ = lgb_style_feval(preds, dmatrix)
        return name, value

    return feval


def _suggest_params(trial: optuna.Trial) -> dict:
    num_leaves = trial.suggest_int("num_leaves", 15, 127)
    return {
        "booster": "gbtree",
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True),
        "num_leaves": num_leaves,
        "max_leaves": num_leaves,
        "grow_policy": "lossguide",
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 500),
        "min_child_weight": trial.suggest_int("min_child_samples", 20, 500),
        "colsample_bytree": trial.suggest_float("feature_fraction", 0.7, 1.0),
        "subsample": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "base_score": 0.0,   # custom objective: start margins at 0
        "verbosity": 0,
    }


def _params_for_xgb(raw_params: dict) -> dict:
    """Strip Optuna/LightGBM-named duplicate keys, keep only xgboost-native ones."""
    return {
        "booster": "gbtree",
        "learning_rate": raw_params["learning_rate"],
        "max_leaves": raw_params["num_leaves"],
        "grow_policy": "lossguide",
        "max_depth": raw_params["max_depth"],
        "min_child_weight": raw_params["min_child_samples"],
        "colsample_bytree": raw_params["feature_fraction"],
        "subsample": raw_params["bagging_fraction"],
        "reg_alpha": raw_params["lambda_l1"],
        "reg_lambda": raw_params["lambda_l2"],
        "base_score": 0.0,
        "verbosity": 0,
    }


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", required=True)
    parser.add_argument("--val-data-dir", default=None)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--n-trials", type=int, default=50)
    parser.add_argument("--timeout", type=float, default=None, help="seconds, optional wall-clock budget")
    parser.add_argument("--num-boost-round", type=int, default=1000, help="cap per trial; early stopping usually triggers first")
    parser.add_argument("--early-stopping", type=int, default=50)
    parser.add_argument("--val-ratio", type=float, default=0.1)
    parser.add_argument("--pos-neg-ratio", type=float, default=5.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--feature-indices", default=None,
                         help="Comma-separated column indices into the stored 15-dim feature "
                              "vector (see yolox/DMA/features.py) to keep, e.g. '1,2,3,7,10,13'. "
                              "Omit to use all 15 features.")
    args = parser.parse_args(argv)

    feature_indices = (
        [int(i) for i in args.feature_indices.split(",")] if args.feature_indices else None
    )
    if feature_indices:
        print(f"Using feature subset (indices={feature_indices}), input_dim={len(feature_indices)}")

    data_paths = sorted(Path(args.data_dir).glob("*.npz"))
    if not data_paths:
        raise FileNotFoundError(f"No .npz files found in {args.data_dir}")

    if args.val_data_dir:
        val_paths = sorted(Path(args.val_data_dir).glob("*.npz"))
        train_ds = DMADataset(
            [str(p) for p in data_paths], normalize=True, pos_neg_ratio=args.pos_neg_ratio,
            feature_indices=feature_indices,
        )
        val_ds = DMADataset([str(p) for p in val_paths], normalize=False, feature_indices=feature_indices)
        val_ds.features = (val_ds.features - train_ds.mean) / train_ds.std
        val_ds.mean, val_ds.std = train_ds.mean, train_ds.std
    else:
        train_ds, val_ds = DMADataset.split(
            [str(p) for p in data_paths], val_ratio=args.val_ratio,
            normalize=True, pos_neg_ratio=args.pos_neg_ratio, seed=args.seed,
            feature_indices=feature_indices,
        )
    print(f"Train: {len(train_ds)} samples  Val: {len(val_ds)} samples")

    dtrain = xgb.DMatrix(train_ds.features, label=train_ds.labels)
    dval = xgb.DMatrix(val_ds.features, label=val_ds.labels)

    train_objective = make_bce_fused_objective(train_ds.motion_costs, train_ds.appearance_costs, train_ds.labels)
    val_feval = _as_xgb_feval(
        make_fused_eval(val_ds.motion_costs, val_ds.appearance_costs, val_ds.labels, name="val_f1")
    )

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)  # create up front so progress is visible immediately

    def _save_progress_callback(study: optuna.Study, trial: optuna.trial.FrozenTrial):
        # Persist best-so-far after every trial so a crash/interrupt doesn't lose all progress
        with open(out_dir / "best_xgb_params.json", "w") as f:
            json.dump({
                "val_f1": study.best_value,
                "params": study.best_params,
                "trials_completed": len(study.trials),
            }, f, indent=2)

    def objective(trial: optuna.Trial) -> float:
        params = _suggest_params(trial)
        evals_result = {}
        booster = xgb.train(
            params, dtrain,
            num_boost_round=args.num_boost_round,
            evals=[(dval, "valid_0")],
            obj=train_objective,
            custom_metric=val_feval,
            maximize=True,
            early_stopping_rounds=args.early_stopping,
            verbose_eval=False,
            evals_result=evals_result,
        )
        trial.set_user_attr("best_iteration", booster.best_iteration)
        return evals_result["valid_0"]["val_f1"][booster.best_iteration]

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=args.seed))
    study.optimize(objective, n_trials=args.n_trials, timeout=args.timeout, callbacks=[_save_progress_callback])

    print(f"\nBest val_f1: {study.best_value:.4f}")
    print(f"Best params: {json.dumps(study.best_params, indent=2)}")

    # Retrain a final model with the best params (generous round budget, same early stopping)
    final_params = _params_for_xgb(study.best_params)
    evals_result = {}
    booster = xgb.train(
        final_params, dtrain,
        num_boost_round=max(args.num_boost_round, study.best_trial.user_attrs["best_iteration"] + args.early_stopping),
        evals=[(dval, "valid_0")],
        obj=train_objective,
        custom_metric=val_feval,
        maximize=True,
        early_stopping_rounds=args.early_stopping,
        verbose_eval=25,
        evals_result=evals_result,
    )

    stats = {
        "mean": train_ds.mean.tolist(),
        "std": train_ds.std.tolist(),
        "feature_indices": feature_indices if feature_indices is not None else list(range(15)),
    }
    best_ckpt = str(out_dir / "dma_xgb_tuned.xgb")
    truncated = booster[: booster.best_iteration + 1]
    DynamicWeightXGB(truncated).save(best_ckpt, stats=stats)
    print(f"Final tuned checkpoint: {best_ckpt}")


if __name__ == "__main__":
    main(shlex.split(XGB_TUNE_ARGS))


# SKlearn

In [ ]:
"""
scikit-learn wrapper for DMA fusion-weight prediction (RandomForest / LogisticRegression).

model_gbm.py / model_xgb.py optimise the *exact* same fused-cost BCE
objective as train.py's DynamicWeightNet, via autograd or a custom booster
objective (see gbm_objective.py). Plain sklearn estimators don't expose a
custom-objective hook generic enough for that, so this module instead uses
one of two proxy training signals (see train_sklearn.py):

  rf      RandomForestRegressor fit with MSE onto a closed-form per-sample
          target weight (soft_weight_target below) -- the exact minimiser of
          that row's fused-cost error.
  logreg  LogisticRegression fit on the binarised version of that same
          target ("should motion dominate for this pair, yes/no"); at
          inference we use predict_proba (a smooth [0, 1] value) rather than
          the hard class label, so w_motion stays continuous.

Same external contract as DynamicWeightNet / DynamicWeightGBM / DynamicWeightXGB:
  predict_numpy(x_np) -> (N, 2) [w_motion, w_reid], rows sum to 1
  save(path, stats) / load(path)
"""

import pickle

import numpy as np


def soft_weight_target(
    motion_cost: np.ndarray, appearance_cost: np.ndarray, labels: np.ndarray, eps: float = 1e-3
) -> np.ndarray:
    """Closed-form per-sample w_motion that makes the fused cost hit (1 - label) exactly."""
    target_fused = 1.0 - labels.astype(np.float64)
    mc = motion_cost.astype(np.float64)
    ac = appearance_cost.astype(np.float64)
    d = mc - ac
    # guard the near-degenerate case where motion and appearance costs agree
    # (any weight gives ~the same fused cost, so the target is ill-defined)
    safe_d = np.where(np.abs(d) < eps, np.where(d >= 0, eps, -eps), d)
    w = (target_fused - ac) / safe_d
    return np.clip(w, 0.0, 1.0).astype(np.float32)


def predict_w_motion(estimator, x_np: np.ndarray) -> np.ndarray:
    """predict_proba(class=1) for classifiers (e.g. LogisticRegression), else predict()."""
    if hasattr(estimator, "predict_proba"):
        w_motion = estimator.predict_proba(x_np)[:, 1]
    else:
        w_motion = estimator.predict(x_np)
    return np.clip(w_motion, 0.0, 1.0).astype(np.float32)


class DynamicWeightSklearn:
    """Wraps a fitted sklearn regressor/classifier predicting w_motion."""

    def __init__(self, estimator, algo: str):
        self.estimator = estimator
        self.algo = algo

    # torch-style no-ops so DMAFusion can treat this like DynamicWeightNet
    def to(self, device):
        return self

    def eval(self):
        return self

    def predict_numpy(self, x_np: np.ndarray) -> np.ndarray:
        w_motion = predict_w_motion(self.estimator, x_np)
        w_reid = 1.0 - w_motion
        return np.stack([w_motion, w_reid], axis=1)

    def save(self, path: str, stats: dict = None):
        payload = {"estimator": self.estimator, "algo": self.algo, "stats": stats}
        with open(path, "wb") as f:
            pickle.dump(payload, f)

    @classmethod
    def load(cls, path: str):
        with open(path, "rb") as f:
            payload = pickle.load(f)
        stats = payload.get("stats", None)
        return cls(payload["estimator"], payload["algo"]), stats


In [ ]:
SKLEARN_TRAIN_ARGS = [
    f"--data-dir {TRAIN_DIR} --val-data-dir {VAL_DIR} "
    f"--out-dir {OUT_ROOT}/sklearn_rf --algo rf "
    f"--feature-indices {FEATURE_INDICES}",
    f"--data-dir {TRAIN_DIR} --val-data-dir {VAL_DIR} "
    f"--out-dir {OUT_ROOT}/sklearn_logreg --algo logreg "
    f"--feature-indices {FEATURE_INDICES}",
]


In [ ]:
"""
Train a scikit-learn RandomForest or LogisticRegression as a DMA
fusion-weight predictor.

Uses the same .npz data pipeline (DMADataset) as train.py / train_gbm.py /
train_xgb.py, but a different training signal: see model_sklearn.py for why
(no generic custom-objective hook in sklearn).
  rf      MSE regression onto a closed-form per-sample target weight.
  logreg  Classification on the binarised target weight; predict_proba
          gives a continuous [0, 1] w_motion at inference.
Treat comparisons against MLP/GBM/XGB as "does the predictor family
matter" rather than a perfectly controlled loss-for-loss ablation.

CLI usage (outside Kaggle):
  python -m yolox.DMA.train_sklearn \\
    --data-dir data/dma_train \\
    --out-dir  weights/dma_sklearn_rf \\
    --algo     rf

In this notebook, main() takes an explicit argv list instead of reading
sys.argv, and runs once per entry in SKLEARN_TRAIN_ARGS (see the cell above)
so both rf and logreg get trained back to back.
"""

import argparse
import json

import numpy as np


def build_estimator(algo: str, args):
    if algo == "rf":
        from sklearn.ensemble import RandomForestRegressor
        return RandomForestRegressor(
            n_estimators=args.n_estimators,
            max_depth=args.max_depth if args.max_depth > 0 else None,
            min_samples_leaf=args.min_child_samples,
            n_jobs=-1, random_state=42,
        )
    if algo == "logreg":
        from sklearn.linear_model import LogisticRegression
        return LogisticRegression(C=args.C, max_iter=args.max_iter, n_jobs=-1)
    raise ValueError(f"Unknown --algo {algo}")


def fit_target(algo: str, motion_cost, appearance_cost, labels):
    soft = soft_weight_target(motion_cost, appearance_cost, labels)
    if algo == "logreg":
        return (soft >= 0.5).astype(np.int64)  # "should motion dominate for this pair"
    return soft


def evaluate(estimator, val_ds) -> float:
    """F1 on the fused-cost < 0.5 decision, matching evaluate() in train.py."""
    w_motion = predict_w_motion(estimator, val_ds.features)
    fused = w_motion * val_ds.motion_costs + (1.0 - w_motion) * val_ds.appearance_costs
    pred_match = (fused < 0.5).astype(np.float64)
    lbl = val_ds.labels.astype(np.float64)

    tp = np.sum((pred_match == 1) & (lbl == 1))
    fp = np.sum((pred_match == 1) & (lbl == 0))
    fn = np.sum((pred_match == 0) & (lbl == 1))
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    return float(2 * precision * recall / (precision + recall + 1e-8))


def train(args):
    data_paths = sorted(Path(args.data_dir).glob("*.npz"))
    if not data_paths:
        raise FileNotFoundError(f"No .npz files found in {args.data_dir}")
    print(f"Found {len(data_paths)} train sequences")

    feature_indices = (
        [int(i) for i in args.feature_indices.split(",")] if args.feature_indices is not None else None
    )
    if feature_indices:
        print(f"Using feature subset (indices={feature_indices})")

    if args.val_data_dir:
        val_paths = sorted(Path(args.val_data_dir).glob("*.npz"))
        if not val_paths:
            raise FileNotFoundError(f"No .npz files found in {args.val_data_dir}")
        print(f"Found {len(val_paths)} val sequences")
        train_ds = DMADataset(
            [str(p) for p in data_paths], normalize=True, pos_neg_ratio=args.pos_neg_ratio,
            feature_indices=feature_indices,
        )
        val_ds = DMADataset([str(p) for p in val_paths], normalize=False, feature_indices=feature_indices)
        val_ds.features = (val_ds.features - train_ds.mean) / train_ds.std
        val_ds.mean = train_ds.mean
        val_ds.std = train_ds.std
    else:
        train_ds, val_ds = DMADataset.split(
            [str(p) for p in data_paths],
            val_ratio=args.val_ratio, normalize=True, pos_neg_ratio=args.pos_neg_ratio,
            feature_indices=feature_indices,
        )

    pos, neg = train_ds.class_balance()
    print(f"Train: {len(train_ds)} samples  pos={pos}  neg={neg}  ratio=1:{neg // max(pos, 1)}")
    print(f"Val:   {len(val_ds)} samples")

    if train_ds.motion_costs is None or train_ds.appearance_costs is None:
        raise ValueError(
            "motion_costs/appearance_costs missing from .npz data -- "
            "required to build the soft-weight training target."
        )

    target = fit_target(args.algo, train_ds.motion_costs, train_ds.appearance_costs, train_ds.labels)
    estimator = build_estimator(args.algo, args)

    print(f"Fitting {args.algo} on {train_ds.features.shape[0]} samples...")
    estimator.fit(train_ds.features, target)

    val_f1 = evaluate(estimator, val_ds)
    print(f"Val fused-cost F1: {val_f1:.4f}")

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stats = {
        "mean": train_ds.mean.tolist(),
        "std": train_ds.std.tolist(),
        "feature_indices": feature_indices if feature_indices is not None else list(range(15)),
    }
    ckpt = str(out_dir / f"dma_sklearn_{args.algo}.skl")
    DynamicWeightSklearn(estimator, args.algo).save(ckpt, stats=stats)
    with open(out_dir / "normalization_stats.json", "w") as f:
        json.dump(stats, f, indent=2)
    print(f"Saved checkpoint: {ckpt}")


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", required=True)
    parser.add_argument("--val-data-dir", default=None)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--algo", required=True, choices=["rf", "logreg"])
    parser.add_argument("--val-ratio", type=float, default=0.1)
    parser.add_argument("--pos-neg-ratio", type=float, default=5.0)
    # rf
    parser.add_argument("--n-estimators", type=int, default=300)
    parser.add_argument("--max-depth", type=int, default=-1, help="<=0 means unlimited")
    parser.add_argument("--min-child-samples", type=int, default=20, help="min_samples_leaf")
    # logreg
    parser.add_argument("--C", type=float, default=1.0, help="LogisticRegression inverse regularisation")
    parser.add_argument("--max-iter", type=int, default=1000)
    parser.add_argument("--feature-indices", default=None,
                         help="Comma-separated column indices into the 15-dim feature vector "
                              "(see yolox/DMA/features.py); default keeps all 15.")
    args = parser.parse_args(argv)
    train(args)


if __name__ == "__main__":
    for _a in SKLEARN_TRAIN_ARGS:
        main(shlex.split(_a))


In [ ]:
SKLEARN_TUNE_ARGS = [
    f"--data-dir {TRAIN_DIR} --val-data-dir {VAL_DIR} "
    f"--out-dir {OUT_ROOT}/sklearn_rf --algo rf "
    f"--n-trials 50 --feature-indices {FEATURE_INDICES}",
    f"--data-dir {TRAIN_DIR} --val-data-dir {VAL_DIR} "
    f"--out-dir {OUT_ROOT}/sklearn_logreg --algo logreg "
    f"--n-trials 50 --feature-indices {FEATURE_INDICES}",
]


In [ ]:
"""
Bayesian hyperparameter search (Optuna, TPE) for DynamicWeightSklearn
(RandomForestRegressor / LogisticRegression).

Uses the exact same data pipeline and soft-weight training target as
train_sklearn.py -- only the estimator hyperparameters are searched, so
results stay comparable with tune_gbm.py / tune_xgb.py's BO-tuned checkpoints.

CLI usage (outside Kaggle):
  python -m yolox.DMA.tune_sklearn \\
    --data-dir  datasets/mot17_dma \\
    --out-dir   weights/dma_sklearn_rf_mot17 \\
    --algo      rf \\
    --n-trials  50

In this notebook, main() takes an explicit argv list instead of reading
sys.argv, and runs once per entry in SKLEARN_TUNE_ARGS (see the cell above)
so both rf and logreg get tuned back to back. fit_target/evaluate are the
ones defined in the train_sklearn.py cell above (same kernel namespace).
"""

import argparse
import json

import optuna


def _suggest_rf(trial: optuna.Trial) -> dict:
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 100),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
        "n_jobs": -1,
        "random_state": 42,
    }


def _suggest_logreg(trial: optuna.Trial) -> dict:
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
    solver = "liblinear" if penalty == "l1" else "lbfgs"
    return {
        "C": trial.suggest_float("C", 1e-4, 1e3, log=True),
        "penalty": penalty,
        "solver": solver,
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "max_iter": 2000,
    }


def _build(algo: str, params: dict):
    if algo == "rf":
        from sklearn.ensemble import RandomForestRegressor
        return RandomForestRegressor(**params)
    if algo == "logreg":
        from sklearn.linear_model import LogisticRegression
        return LogisticRegression(**params)
    raise ValueError(f"Unknown --algo {algo}")


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", required=True)
    parser.add_argument("--val-data-dir", default=None)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--algo", required=True, choices=["rf", "logreg"])
    parser.add_argument("--n-trials", type=int, default=50)
    parser.add_argument("--timeout", type=float, default=None, help="seconds, optional wall-clock budget")
    parser.add_argument("--val-ratio", type=float, default=0.1)
    parser.add_argument("--pos-neg-ratio", type=float, default=5.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--feature-indices", default=None,
                         help="Comma-separated column indices into the stored 15-dim feature "
                              "vector (see yolox/DMA/features.py) to keep, e.g. '1,2,3,7,10,13'. "
                              "Omit to use all 15 features.")
    args = parser.parse_args(argv)

    feature_indices = (
        [int(i) for i in args.feature_indices.split(",")] if args.feature_indices else None
    )
    if feature_indices:
        print(f"Using feature subset (indices={feature_indices}), input_dim={len(feature_indices)}")

    data_paths = sorted(Path(args.data_dir).glob("*.npz"))
    if not data_paths:
        raise FileNotFoundError(f"No .npz files found in {args.data_dir}")

    if args.val_data_dir:
        val_paths = sorted(Path(args.val_data_dir).glob("*.npz"))
        train_ds = DMADataset(
            [str(p) for p in data_paths], normalize=True, pos_neg_ratio=args.pos_neg_ratio,
            feature_indices=feature_indices,
        )
        val_ds = DMADataset([str(p) for p in val_paths], normalize=False, feature_indices=feature_indices)
        val_ds.features = (val_ds.features - train_ds.mean) / train_ds.std
        val_ds.mean, val_ds.std = train_ds.mean, train_ds.std
    else:
        train_ds, val_ds = DMADataset.split(
            [str(p) for p in data_paths], val_ratio=args.val_ratio,
            normalize=True, pos_neg_ratio=args.pos_neg_ratio, seed=args.seed,
            feature_indices=feature_indices,
        )
    print(f"Train: {len(train_ds)} samples  Val: {len(val_ds)} samples")

    if train_ds.motion_costs is None or train_ds.appearance_costs is None:
        raise ValueError(
            "motion_costs/appearance_costs missing from .npz data -- "
            "required to build the soft-weight training target."
        )

    target = fit_target(args.algo, train_ds.motion_costs, train_ds.appearance_costs, train_ds.labels)

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)  # create up front so progress is visible immediately

    def _save_progress_callback(study: optuna.Study, trial: optuna.trial.FrozenTrial):
        # Persist best-so-far after every trial so a crash/interrupt doesn't lose all progress
        with open(out_dir / f"best_sklearn_{args.algo}_params.json", "w") as f:
            json.dump({
                "val_f1": study.best_value,
                "params": study.best_params,
                "trials_completed": len(study.trials),
            }, f, indent=2)

    suggest_fn = _suggest_rf if args.algo == "rf" else _suggest_logreg

    def objective(trial: optuna.Trial) -> float:
        params = suggest_fn(trial)
        estimator = _build(args.algo, params)
        estimator.fit(train_ds.features, target)
        return evaluate(estimator, val_ds)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=args.seed))
    study.optimize(objective, n_trials=args.n_trials, timeout=args.timeout, callbacks=[_save_progress_callback])

    print(f"\nBest val_f1: {study.best_value:.4f}")
    print(f"Best params: {json.dumps(study.best_params, indent=2)}")

    # Refit the final estimator with the best params
    if args.algo == "logreg":
        best_params = dict(study.best_params)
        best_params["solver"] = "liblinear" if best_params["penalty"] == "l1" else "lbfgs"
        best_params["max_iter"] = 2000
    else:
        best_params = dict(study.best_params)
        best_params.update({"n_jobs": -1, "random_state": 42})
    estimator = _build(args.algo, best_params)
    estimator.fit(train_ds.features, target)
    val_f1 = evaluate(estimator, val_ds)
    print(f"Refit val_f1: {val_f1:.4f}")

    stats = {
        "mean": train_ds.mean.tolist(),
        "std": train_ds.std.tolist(),
        "feature_indices": feature_indices if feature_indices is not None else list(range(15)),
    }
    best_ckpt = str(out_dir / f"dma_sklearn_{args.algo}_tuned.skl")
    DynamicWeightSklearn(estimator, args.algo).save(best_ckpt, stats=stats)
    print(f"Final tuned checkpoint: {best_ckpt}")


if __name__ == "__main__":
    for _a in SKLEARN_TUNE_ARGS:
        main(shlex.split(_a))
